In [2]:
import numpy as np
import tensorflow as tf

In [9]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)


def rnn_cell_forward(xt, a_prev, parameters):
  
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]
    
    a_next = tanh(np.dot(Wax, xt) + np.dot(Waa, a_prev) + ba)
    yt_pred = softmax(np.dot(Wya, a_next) + by)
    
    cache = (a_next, a_prev, xt, parameters)
    
    return a_next, yt_pred, cache

In [22]:
def cell_forward(X, A, params):
    
    WA = tf.linalg.matmul(a=params["Waa"], b=A, name="WA forward")
    WX = tf.linalg.matmul(a=params["Wax"], b=X, name="WX forward")
    A_ = tf.math.tanh(WX + WA + params["ba"], name="A_ forward")
    WY = tf.linalg.matmul(a=params["Wya"], b=A_, name="WY forward")
    Y_hat = tf.keras.activations.softmax(x=WY + params["by"], axis=0)

    cache = (A_, A, X, params)

    return (A_, Y_hat, cache)    

In [23]:
np.random.seed(1)
xt = np.random.randn(3, 1)      
a_prev = np.random.randn(5, 1)     


parameters = {
    "Wax": np.random.randn(5, 3),
    "Waa": np.random.randn(5, 5),   
    "Wya": np.random.randn(2, 5),  
    "ba": np.random.randn(5, 1),   
    "by": np.random.randn(2, 1)    
}

a_next, yt_pred, cache = rnn_cell_forward(xt, a_prev, parameters)


print("Next hidden state a_next:\n", a_next)
print("\nPredicted output yt_pred:\n", yt_pred)

Next hidden state a_next:
 [[ 0.98651002]
 [ 0.10754463]
 [ 0.07820302]
 [-0.97496223]
 [-0.99935928]]

Predicted output yt_pred:
 [[0.95151021]
 [0.04848979]]


In [24]:
a, y, cache = cell_forward(X=xt, A=a_prev, params=parameters)
print(a)
print(y)

tf.Tensor(
[[ 0.98651002]
 [ 0.10754463]
 [ 0.07820302]
 [-0.97496223]
 [-0.99935928]], shape=(5, 1), dtype=float64)
tf.Tensor(
[[0.95151021]
 [0.04848979]], shape=(2, 1), dtype=float64)


In [25]:
def rnn_forward(x, a0, parameters):

    caches = []
    
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wya"].shape
    
    a = np.zeros((n_a, m, T_x), dtype=float)
    y_pred = np.zeros((n_y, m, T_x), dtype=float)
    
    a_next = a0
    
    for t in range(T_x):
    
        a_next, yt_pred, cache = rnn_cell_forward(xt=x[:, :, t], a_prev=a_next, parameters=parameters)
        a[:,:,t] = a_next
        y_pred[:,:,t] = yt_pred
        caches.append(cache)
    
    caches = (caches, x)
    
    return a, y_pred, caches

In [41]:
def cells_forward(X, A0, params):
    caches = []
    
    n_x, m, t_x = X.shape
    n_y, n_a = params["Wya"].shape

    # Using TensorArray to keep track of tensor updates in each time step
    A = tf.TensorArray(dtype=tf.float32, size=t_x, clear_after_read=False)
    Y_hat = tf.TensorArray(dtype=tf.float32, size=t_x, clear_after_read=False)

    A_ = A0

    for t in range(t_x):
        # Perform one step of forward pass
        A_, y_hat, cache = cell_forward(X=X[:, :, t], A=A_, params=params)

        # Store the result in the TensorArray
        A = A.write(t, A_)
        Y_hat = Y_hat.write(t, y_hat)

        caches.append(cache)

    # Stack TensorArray into a full tensor
    A = tf.transpose(A.stack(), perm=[1, 2, 0])  # Shape (n_a, m, t_x)
    Y_hat = tf.transpose(Y_hat.stack(), perm=[1, 2, 0])  # Shape (n_y, m, t_x)

    caches = (X, caches)

    return A, Y_hat, caches

In [42]:
np.random.seed(1)
tf.random.set_seed(1)

x = np.random.randn(3, 5, 10)   
X = tf.constant(x, dtype=tf.float32)  

a0 = np.random.randn(5, 5)  
A0 = tf.constant(a0, dtype=tf.float32) 

parameters = {
    "Wax": np.random.randn(5, 3),
    "Waa": np.random.randn(5, 5),
    "Wya": np.random.randn(2, 5),
    "ba": np.random.randn(5, 1),
    "by": np.random.randn(2, 1)
}

params = {k: tf.constant(v, dtype=tf.float32) for k, v in parameters.items()}

a, y, c = rnn_forward(x=x, a0=a0, parameters=parameters)
print(a)
print(y)

A, Y_hat, caches_tf = cells_forward(X=X, A0=A0, params=params)

print(A)
print(Y_hat)



[[[ 0.54250085  0.99713474 -0.15832201  0.93997869  0.99337727
    0.99609951  0.99911314  0.99723066  0.987311    0.99989163]
  [-0.91799325  0.99985395  0.99967119  0.75087995  0.96056688
    0.99987071  0.99708777  0.99976517  0.99997567  0.23354203]
  [ 0.9996474   0.91719989  0.99631262  0.9871928   0.99776057
   -0.92992032 -0.95201695  0.99998043  0.99999428  0.95446991]
  [ 0.99999638  0.99766427  0.92894248  0.98508448  0.99949304
    0.95836518  0.77819293  0.99964379  0.47440659  0.99256737]
  [ 0.99599177  0.99789142  0.92106942  0.99201514  0.99678531
    0.99952982  0.9991472   0.98059513  0.99949912  0.99977091]]

 [[ 0.41496865 -0.99145881 -0.99949978 -0.98955693  0.6320513
   -0.9998778  -0.99953622 -0.98362174 -0.98164217 -0.99917597]
  [ 0.99999954 -0.61925845 -0.99989353 -0.96320451 -0.98883656
   -0.93202919 -0.99262622 -0.98139361 -0.99991221 -0.93296562]
  [ 0.3407794  -0.60255412 -0.99984852 -0.9999304   0.45412094
   -0.99978841 -0.99397997  0.83686635  0.82365